In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm  # 引入tqdm来显示进度条

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ==============================================================================
# --- 1. 配置 (请确认这里的路径) ---
# ==============================================================================
# SAM2 Hydra 配置名和检查点路径
SAM2_CFG_NAME = "configs/sam2.1/sam2.1_hiera_b+"
# !! 注意: 这里使用的是您提供的微调后的检查点路径，这将评估微调后的模型。
# !! 如果要评估原始SAM2的Baseline，请换回原始检查点路径。
SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2_finetune_logs/busi_direct_run_2/checkpoints/checkpoint.pt"
# SAM2_CHECKPOINT_PATH = "/home/zhengsongming/jupyterworkspace/03医学图像分割/sam2/checkpoints/sam2.1_hiera_base_plus.pt" # <--- 原始Baseline用这个


HYDRA_OVERRIDES = [
    # _target_ 是一个已有的字段，我们是在“覆盖”它，所以不需要加号
    "model.image_encoder.trunk._target_=sam2.modeling.backbones.hieradet_adapterv1.Hiera",
    
    # use_adapter 是一个我们新添加的字段，所以前面必须加上 `+`
    "+model.image_encoder.trunk.use_adapter=True",
    
    # 这个 use_adapter 也是新增的，同样加上 `+`
    "+model.use_adapter=True",
]
# 已划分的数据集根目录 (包含JPEGImages, Annotations, ImageSets)
SPLIT_DATASET_ROOT = "/home/zhengsongming/jupyterworkspace/datasets/BUSI_for_SAM2"

# 测试集列表文件路径
TEST_SET_FILE = os.path.join(SPLIT_DATASET_ROOT, "ImageSets", "2017", "val.txt")


# ==============================================================================
# --- 2. 初始化SAM2模型和预测器 (无需修改) ---
# ==============================================================================
print("Loading SAM2 model in Adapter mode...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_sam2(
    SAM2_CFG_NAME, 
    SAM2_CHECKPOINT_PATH, 
    device=device,
    hydra_overrides_extra=HYDRA_OVERRIDES # <-- 将我们的覆盖配置传进去
)
predictor = SAM2ImagePredictor(model)
print(f"Model and Predictor loaded on {device}.")


# ==============================================================================
# --- 3. 定义指标计算函数 (无需修改) ---
# ==============================================================================
def calculate_metrics(gt_mask, pred_mask):
    """计算Dice系数和IoU"""
    gt_mask_bool = gt_mask > 0
    pred_mask_bool = pred_mask > 0
    intersection = np.logical_and(gt_mask_bool, pred_mask_bool).sum()
    dice = (2. * intersection) / (gt_mask_bool.sum() + pred_mask_bool.sum() + 1e-8)
    union = gt_mask_bool.sum() + pred_mask_bool.sum() - intersection
    iou = intersection / (union + 1e-8)
    return dice, iou

# ==============================================================================
# --- 4. 遍历测试集、执行分割并评估 (核心修改部分) ---
# ==============================================================================
# 检查测试集文件是否存在
if not os.path.exists(TEST_SET_FILE):
    raise FileNotFoundError(f"测试集定义文件未找到: {TEST_SET_FILE}")

# 读取测试集样本名称
with open(TEST_SET_FILE, 'r') as f:
    test_sample_names = [line.strip() for line in f.readlines()]

print(f"\nFound {len(test_sample_names)} samples in the test set. Starting evaluation...")

# 用于存储所有结果的列表
results = []
# 定义图像和掩码的根目录
images_dir = os.path.join(SPLIT_DATASET_ROOT, "JPEGImages")
annotations_dir = os.path.join(SPLIT_DATASET_ROOT, "Annotations")


# 使用tqdm来可视化进度
for sample_name in tqdm(test_sample_names, desc="Evaluating Test Set"):
    # 根据新的目录结构构建文件路径
    image_path = os.path.join(images_dir, sample_name, "00000.jpg")
    mask_path = os.path.join(annotations_dir, sample_name, "00000.png")
    
    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        print(f"Warning: Files for sample '{sample_name}' not found. Skipping.")
        continue

    # 读取原始图像和金标准掩码
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # --- 分割流程 (与您原来的代码完全相同) ---
    predictor.set_image(image_rgb)
    contours, _ = cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        print(f"Warning: No lesion found in mask for {sample_name}. Skipping.")
        continue
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    box_prompt = np.array([[x, y, x + w, y + h]])

    masks, scores, logits = predictor.predict(
        box=box_prompt,
        multimask_output=False
    )
    
    pred_mask = (masks[0] * 255).astype(np.uint8)
    
    # --- 立即评估 ---
    dice, iou = calculate_metrics(gt_mask, pred_mask)
    
    # --- 记录结果 ---
    # 从样本名中判断类别
    category = 'benign' if 'benign' in sample_name else 'malignant'
    results.append({
        "category": category,
        "sample_name": sample_name,
        "dice": dice,
        "iou": iou
    })

    # 重置预测器状态
    predictor.reset_predictor()

# ==============================================================================
# --- 5. 打印最终的总结报告 (无需修改) ---
# ==============================================================================
if results:
    df = pd.DataFrame(results)
    
    # 按类别计算平均值
    category_summary = df.groupby('category').agg(
        mean_dice=('dice', 'mean'),
        std_dice=('dice', 'std'),
        mean_iou=('iou', 'mean'),
        std_iou=('iou', 'std'),
        count=('sample_name', 'count')
    ).reset_index()
    
    print("\n" + "="*50)
    print("--- Evaluation Summary on Test Set (val.txt) ---")
    print("="*50)
    print(category_summary.to_string())
    print("="*50)

else:
    print("No images were processed. Please check paths and file structure.")

print("\n--- Evaluation on the test set is complete! ---")

Loading SAM2 model in Adapter mode...
Model and Predictor loaded on cuda.

Found 130 samples in the test set. Starting evaluation...


Evaluating Test Set: 100%|██████████| 130/130 [00:19<00:00,  6.53it/s]


--- Evaluation Summary on Test Set (val.txt) ---
    category  mean_dice  std_dice  mean_iou   std_iou  count
0     benign   0.939956  0.026413  0.887849  0.046049     91
1  malignant   0.907119  0.038260  0.832143  0.062178     39

--- Evaluation on the test set is complete! ---
